# Convolutional Neural Networks (CNNs)
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> Kernels, Strides, Padding, Feature Maps, Pooling, Receptive Fields, and Classic Architectures from LeNet to ResNet.

---

### 📖 Conceptual Foundations

#### Why Fully Connected (Dense) Layers Fail on Images

A standard $1024 \times 1024 \times 3$ RGB color image has $3,145,728$ input values. Connecting this to a single hidden layer of just $1,000$ neurons requires over **$3.14$ billion parameters**! Furthermore, dense layers flatten images into 1D vectors, completely discarding 2D spatial relationships. **CNNs solve this** via two inductive biases:
1. **Local Connectivity:** Each neuron connects only to a small local patch (receptive field) of the input.
2. **Weight Sharing (Parameter Sharing):** The same kernel (filter) sweeps across the entire image, detecting features invariant to their spatial position.


#### The 2D Convolution Dimension Formula

Given an input feature map of height $H_{in}$ and width $W_{in}$, filter/kernel size $K$, padding $P$, and stride $S$, the output dimensions are:

$$H_{out} = \left\lfloor \frac{H_{in} - K + 2P}{S} \right\rfloor + 1$$
$$W_{out} = \left\lfloor \frac{W_{in} - K + 2P}{S} \right\rfloor + 1$$

- **Valid Padding ($P = 0$):** Output shrinks: $H_{out} = H_{in} - K + 1$.
- **Same Padding:** Preserves spatial dimensions ($H_{out} = H_{in}$ when $S=1$) by setting $P = \frac{K - 1}{2}$ for odd kernel sizes.


#### Milestone CNN Architectures Through Time

| Architecture | Year & Authors | Key Architectural Innovations | Significance |
| --- | --- | --- | --- |
| LeNet-5 | 1998 (LeCun et al.) | 5x5 convs, average pooling, tanh activations. | Pioneered handwritten digit recognition (MNIST) for bank checks. |
| AlexNet | 2012 (Krizhevsky et al.) | ReLU activations, Dropout (0.5), GPU training (2x GTX 580), 11x11 & 5x5 filters. | Ignited the modern Deep Learning revolution by winning ImageNet 2012 with a 15.3% top-5 error rate. |
| VGGNet (VGG-16/19) | 2014 (Simonyan & Zisserman) | Stacked homogeneous $3\times 3$ convolutions with stride 1 instead of larger $5\times 5$ or $7\times 7$ filters. | Proved two stacked $3\times 3$ convs have the same receptive field as a $5\times 5$ conv but with 28% fewer parameters and extra non-linearities. |
| GoogLeNet (Inception v1) | 2014 (Szegedy et al.) | Multi-scale Inception modules (parallel $1\times 1, 3\times 3, 5\times 5$ convs + pooling), $1\times 1$ conv dimensionality reduction. | Achieved 22 layers with 12x fewer parameters than AlexNet (4M vs 60M). |
| ResNet (ResNet-50/152) | 2015 (He et al., Microsoft) | Residual skip connections ($y = \mathcal{F}(x) + x$), Bottleneck blocks ($1\times 1 \to 3\times 3 \to 1\times 1$), Batch Normalization. | Solved the gradient degradation problem, enabling ultra-deep networks (152+ layers) and surpassing human-level ImageNet accuracy (3.57% top-5). |


#### Why $1 \times 1$ Convolutions (Pointwise Convolutions) are Powerful

A $1 \times 1$ convolution computes a linear combination across channels at each spatial coordinate $(h, w)$. It enables:
1. **Dimensionality Reduction:** Compressing 256 input channels down to 64 channels before expensive $3 \times 3$ convolutions, slashing FLOPs by 75%.
2. **Added Non-Linearity:** Introducing an extra ReLU activation without changing spatial height/width.



In [ ]:
# %matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def conv2d_forward(X, W, b, stride=1, padding=1):
    # X shape: (N, C_in, H_in, W_in)
    # W shape: (C_out, C_in, K_h, K_w)
    N, C_in, H_in, W_in = X.shape
    C_out, _, K_h, K_w = W.shape
    
    # Pad input
    X_pad = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')
    H_out = int((H_in - K_h + 2 * padding) / stride) + 1
    W_out = int((W_in - K_w + 2 * padding) / stride) + 1
    
    out = np.zeros((N, C_out, H_out, W_out))
    for n in range(N):
        for c in range(C_out):
            for h in range(H_out):
                for w in range(W_out):
                    h_start = h * stride
                    h_end = h_start + K_h
                    w_start = w * stride
                    w_end = w_start + K_w
                    
                    patch = X_pad[n, :, h_start:h_end, w_start:w_end]
                    out[n, c, h, w] = np.sum(patch * W[c, :, :, :]) + b[c]
    return out

# Create a 16x16 synthetic image with a vertical edge
img = np.zeros((1, 1, 16, 16))
img[0, 0, :, 8:] = 1.0 # Left side dark, right side bright

# Sobel vertical edge filter (1, 1, 3, 3)
sobel_filter = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
]).reshape(1, 1, 3, 3)
b = np.array([0.0])

feature_map = conv2d_forward(img, sobel_filter, b, stride=1, padding=1)
print('Output Feature Map Shape:', feature_map.shape)
print('Max Edge Intensity:', np.max(feature_map))

plt.figure(figsize=(7, 3))
plt.subplot(1, 2, 1); plt.imshow(img[0, 0], cmap='gray'); plt.title('Original Image (Edge)')
plt.subplot(1, 2, 2); plt.imshow(feature_map[0, 0], cmap='magma'); plt.title('Detected Vertical Boundary')
plt.tight_layout(); plt.show()


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

